<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">模块 9：数仓建模与 JOIN</div><div style="color:#17212b;font-size:30px;font-weight:750">模块 9：数仓建模与 JOIN</div><p style="color:#475569;line-height:1.7">从单张导入订单表扩展到小型事实表与维表模型。请按顺序运行；结果会以表格展示，写入只作用于本模块的 `_l2` 对象。</p></div>

## 边界

本实验不会修改 Level 1 源表，只创建或替换带 `_l2` 后缀的对象。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab

lab = WarehouseLab()


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_fact_l2")
lab.execute("DROP TABLE IF EXISTS customers_dim_l2")
lab.execute("""
CREATE TABLE customers_dim_l2 (
    customer_id BIGINT NOT NULL,
    customer_name VARCHAR(100) NOT NULL,
    region VARCHAR(32) NOT NULL
)
UNIQUE KEY(customer_id)
DISTRIBUTED BY HASH(customer_id) BUCKETS 1
PROPERTIES ("replication_num"="1", "enable_unique_key_merge_on_write"="true")
""")
lab.execute("""
CREATE TABLE orders_fact_l2 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    order_amount DECIMAL(18,2) NOT NULL,
    data_source VARCHAR(32) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
DISTRIBUTED BY HASH(customer_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.execute("INSERT INTO customers_dim_l2 SELECT customer_id, customer_name, 'known' FROM customers LIMIT 20")
lab.execute("INSERT INTO orders_fact_l2 SELECT DATE(event_time), order_id, customer_id, order_amount, data_source FROM orders_imported")
lab.sql("SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT order_id) AS order_grain FROM orders_fact_l2", title="事实表粒度契约")

In [ ]:
lab.sql("""
SELECT f.order_id, f.order_amount, d.customer_name, d.region
FROM orders_fact_l2 AS f
LEFT JOIN customers_dim_l2 AS d ON d.customer_id = f.customer_id
ORDER BY f.order_id
LIMIT 10
""", title="事实表与客户维表")
lab.sql("""
SELECT f.customer_id, COUNT(*) AS orders_without_dimension
FROM orders_fact_l2 AS f
LEFT JOIN customers_dim_l2 AS d ON d.customer_id = f.customer_id
WHERE d.customer_id IS NULL
GROUP BY f.customer_id
ORDER BY f.customer_id
""", title="缺失维度的反连接检查")

In [ ]:
lab.sql("""
SELECT d.region, SUM(f.order_amount) AS amount
FROM orders_fact_l2 AS f
JOIN customers_dim_l2 AS d ON d.customer_id = f.customer_id
GROUP BY d.region
ORDER BY d.region
""", title="按维度扩展的指标")
lab.sql("""
EXPLAIN
SELECT f.order_id, d.customer_name
FROM orders_fact_l2 AS f
JOIN customers_dim_l2 AS d ON d.customer_id = f.customer_id
""", title="JOIN 分布计划")

## 要点

将结果与课程中声明的业务粒度对照。SQL 执行成功本身不能证明模型、指标、访问边界或消费者契约正确。